# 28_dcgan.ipynb

**13주차 · 3교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`13week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 8셀. 위에서부터 순서대로 실행합니다.

## 2-1. `ConvTranspose2d` — Conv 의 반대 방향

**셀 1** — 크기가 정말 커지는지 확인 (30초)

In [ ]:
import torch, torch.nn as nn
x = torch.randn(1, 64, 7, 7)
up = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
print("입력 :", x.shape, "→ 출력 :", up(x).shape)     # (1, 32, 14, 14)  ★ 2배

## 2-2. 구조 읽기

**셀 2** — 생성자 · 판별자 (배포본. 읽기 중심 ★)

In [ ]:
NZ = 100                                   # 잠재 벡터 차원

class Generator(nn.Module):                # (B,100,1,1) → (B,1,28,28)
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ, 128, 7, 1, 0, bias=False),  # → (B,128,7,7)
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),  # → (B,64,14,14)
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),    # → (B,1,28,28)
            nn.Tanh(),                                          # ★ 출력 [-1,1]
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):            # (B,1,28,28) → (B,1)
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 1, 7, 1, 0, bias=False),             # → (B,1,1,1)
        )
    def forward(self, x): return self.net(x).view(-1)           # logits

G, D = Generator(), Discriminator()
print("G :", G(torch.randn(2, NZ, 1, 1)).shape)     # (2,1,28,28)
print("D :", D(torch.randn(2, 1, 28, 28)).shape)    # (2,)

## 2-3. 짧은 학습 (실행 5~10분)

**셀 3** — 학습 (배포본 실행 ★)

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.5,), (0.5,))])   # ★ [-1,1]
dl = DataLoader(datasets.FashionMNIST("data", train=True, transform=tf),
                batch_size=128, shuffle=True, drop_last=True)

G, D = Generator().to(DEV), Discriminator().to(DEV)
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce  = nn.BCEWithLogitsLoss()
EPOCHS = 5                                  # ★ 안내되는 값으로 조정

for ep in range(1, EPOCHS + 1):
    for xb, _ in dl:
        xb = xb.to(DEV); b = xb.size(0)
        # ① D : 진짜는 1, 가짜는 0
        z = torch.randn(b, NZ, 1, 1, device=DEV)
        fake = G(z)
        lossD = bce(D(xb), torch.ones(b, device=DEV)) + \
                bce(D(fake.detach()), torch.zeros(b, device=DEV))   # ★ detach
        optD.zero_grad(); lossD.backward(); optD.step()
        # ② G : 내 가짜를 1 로 봐 달라
        lossG = bce(D(fake), torch.ones(b, device=DEV))
        optG.zero_grad(); lossG.backward(); optG.step()
    print(f"epoch {ep} | D {lossD.item():.3f} | G {lossG.item():.3f}")

> **관찰 포인트 ★**: **손실이 내려가지 않습니다.** 오르내립니다. 분류기와 달리 **손실 값으로는 잘 되는지 알 수 없습니다.** *"그래서 GAN 은 눈으로 봐야 한다"* 를 여기서 체감하게 됩니다.

**셀 4** — 결과 격자

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

FIXED_Z = torch.randn(64, NZ, 1, 1, device=DEV)      # ★ 고정 z (6주차 재현성)
G.eval()
with torch.no_grad(): imgs = G(FIXED_Z).cpu()
plt.figure(figsize=(7, 7))
plt.imshow(make_grid(imgs, nrow=8, normalize=True).permute(1, 2, 0), cmap="gray")
plt.axis("off"); plt.title(f"짧은 학습 {EPOCHS} epoch — 흐릿한 것이 정상 ★")
plt.savefig("outputs/dcgan_short.png", dpi=120, bbox_inches="tight"); plt.show()

## 3. 실습 7 — 사전학습 가중치 결과와 비교 ★

**셀 5** — 배포 가중치 (같은 코드, 오래 학습) ★

In [ ]:
G_pre = Generator().to(DEV)
G_pre.load_state_dict(torch.load("models/dcgan_pretrained.pt", map_location=DEV))
G_pre.eval()
with torch.no_grad(): imgs_pre = G_pre(FIXED_Z).cpu()   # ★ 같은 z 를 쓴다

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
axes[0].imshow(make_grid(imgs,     nrow=8, normalize=True).permute(1,2,0), cmap="gray")
axes[0].set_title(f"짧은 학습 ({EPOCHS} epoch)"); axes[0].axis("off")
axes[1].imshow(make_grid(imgs_pre, nrow=8, normalize=True).permute(1,2,0), cmap="gray")
axes[1].set_title("충분한 학습 (50~100 epoch)");  axes[1].axis("off")
plt.suptitle("같은 코드 · 같은 데이터 · 같은 z — 차이는 학습량뿐 ★★")
plt.savefig("outputs/dcgan_compare.png", dpi=120, bbox_inches="tight"); plt.show()

> **핵심 ★★**: **코드도 데이터도 잠재 벡터도 같습니다. 다른 것은 학습 시간뿐입니다.** *"생성 모델에서는 **학습 시간이 곧 자원**이다"* 라는 감각을 여기서 심어 주세요. 이것이 **왜 큰 생성 모델을 개인이 못 만드는가**에 대한 답이기도 합니다.

**셀 6** — 생성 이미지를 어떻게 평가할 것인가 (토의)

In [ ]:
print("""
  분류기 :  정확도 87.3%      →  숫자 하나로 비교된다
  생성기 :  ???                →  "그럴듯한가"를 어떻게 숫자로?

    실무 지표 : FID · IS (사전학습 분류기의 특징 분포를 비교)
    한계      : 사람이 보기에 좋은 것과 늘 일치하지는 않는다  ★
    → 그래서 생성 모델 논문에는 항상 "샘플 이미지"가 실린다
""")

> **핵심 ★ (기말 출제 지점)**: **생성 모델에는 정확도가 없습니다.** 이것이 생성 모델을 평가하기 어려운 근본 이유이고, 미니 프로젝트에서 생성 주제를 고른다면 **평가 방법을 미리 정해 두어야** 합니다.

## 4. 실습 8 — 잠재 벡터 조작

**셀 7** — z 를 바꾸면 출력이 어떻게 변하나

In [ ]:
z1 = torch.randn(1, NZ, 1, 1, device=DEV)
z2 = torch.randn(1, NZ, 1, 1, device=DEV)
alphas = torch.linspace(0, 1, 8, device=DEV).view(-1, 1, 1, 1)
zs = (1 - alphas) * z1 + alphas * z2                       # ★ 2교시 보간과 같은 코드
with torch.no_grad(): imgs_i = G_pre(zs).cpu()

fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for k in range(8):
    axes[k].imshow(imgs_i[k, 0], cmap="gray"); axes[k].axis("off")
plt.suptitle("GAN 잠재 공간 보간 — 오토인코더와 같은 성질 ★")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: **오토인코더에서 본 것과 같은 일**이 일어납니다. 잠재 공간이 연속적이고 의미를 담고 있다는 성질은 **AE·VAE·GAN 에 공통**입니다. 그리고 **14주차 Stable Diffusion 의 seed** 가 바로 이 z 에 해당합니다.

**셀 8** — seed 를 고정하면 같은 이미지 (6주차 재현성 ★)

In [ ]:
for s in [0, 0, 1]:
    torch.manual_seed(s)
    z = torch.randn(1, NZ, 1, 1, device=DEV)
    with torch.no_grad(): im = G_pre(z).cpu()
    print(f"seed={s} → 픽셀 합 {im.sum().item():.4f}")
print("→ 같은 seed = 같은 이미지. 14주차 SD 에서도 똑같다 ★")